<a href="https://colab.research.google.com/github/monasolgi/Daily_Climate_time_series_data/blob/main/Daily_Climate_time_series_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

#====================================
#Load Data
data=pd.read_csv('/content/DailyDelhiClimateTrain.csv', index_col='date' )
data.info()
data.head(5)


In [ ]:
data.describe()
data.dtypes

In [ ]:
df=data.copy()
df.isna().sum()

In [ ]:
df.columns
df['meantemp']

In [ ]:
from numpy import datetime64
df.index
print(df.index.is_monotonic_increasing)
print(df.index.duplicated().sum())
print(df.index.dtype)

#convert index into datetime
df.index=pd.to_datetime(df.index)

In [ ]:
print(df.index.dtype)


In [ ]:
from matplotlib import figure
#Visualinging columns
import matplotlib.pyplot as plt

fig,ax=plt.subplots(nrows=2,ncols=2,figsize=(10,5))
ax=ax.flatten()

for i,col in enumerate(df.columns):
  ax[i].plot(df.index,df[col])
  ax[i].set_title(col)

fig.autofmt_xdate() #to rotate labels
plt.tight_layout()
plt.show()

#meantemp shows strong yearly seasonality


In [ ]:
print(df['meanpressure'].describe())
#print(df['meanpressure'].sort_values().head(20))
#print(df['meanpressure'].sort_values().tail(20))
df['meanpressure'].hist(bins=100)

In [ ]:
#Convert impossible values to NaN
import numpy as np

q1=df['meanpressure'].quantile(0.25)
q3=df['meanpressure'].quantile(0.75)

#Data points that fall below Q1−1.5×IQR or above Q3+1.5×IQR are considered outliers
IQR=q3 - q1
lower=q1 - 1.5 * IQR
upper=q3 + 1.5 * IQR

df.loc[ (df['meanpressure']< lower) | (df['meanpressure']>upper)]

df.loc[ (df['meanpressure']< lower) | (df['meanpressure']>upper),"meanpressure"]=np.nan
print(df.isnull().sum())

#interpolate
df["meanpressure"]=df["meanpressure"].interpolate(method="time")
print(df.isnull().sum())
df['meanpressure'].hist(bins=100)

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(df.index,df["meanpressure"])
plt.title("Mean Pressure After Outlier Cleaning")
plt.show()

In [ ]:
import seaborn as sns
plt.figure(figsize=(5,3))
sns.heatmap(df.corr(),annot=True)
plt.show()

#captures only linear relation
#absolute value od relation is considered
#therefore highest correlattion is 0.88 between target and meanpressure

In [ ]:
#Temperature lags
target=df["meantemp"]

df['lag_1']=target.shift(1)
df['lag_7']=target.shift(7)
df['lag_14']=target.shift(14)
df['lag_30']=target.shift(30)

#Temperature rolling statistics
df['rolling_mean_7']=target.rolling(7).mean()
df['rolling_mean_30']=target.rolling(30).mean()
df['rolling_std_7']=target.rolling(7).std()

#calender features
df['day_of_week']=df.index.day_of_week
df['day_of_year']=df.index.day_of_year
df['month']=df.index.month

print(df.columns)

In [ ]:
print(df.isnull().sum())

#drop nans created by new features
df=df.dropna()
print(df.isnull().sum())

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

xgb_reg=XGBRegressor(

    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    random_state=42

)

tscv=TimeSeriesSplit(gap=0,n_splits=5)

mae_scores=[]
rmse_scores=[]
r2_scores=[]

X=df.drop(columns=['meantemp'])
y=target

for i,(train_index,test_index) in enumerate(tscv.split(X)):

  x_train=X.iloc[train_index]
  y_train=y.iloc[train_index]

  x_test=X.iloc[test_index]
  y_test=y.iloc[test_index]

  xgb_model=xgb_reg.fit(x_train,y_train)
  y_pred=xgb_model.predict(x_test)

  mae=mean_absolute_error(y_test,y_pred)
  rmse=root_mean_squared_error(y_test,y_pred)
  r2=r2_score(y_test,y_pred)

  mae_scores.append(mae)
  rmse_scores.append(rmse)
  r2_scores.append(r2)

  print(f"Fold {i+1}")
  print("MAE:", mae)
  print("RMSE:", rmse)
  print("R2:", r2)
  print("-"*30)

print("\nAverage Results")
print("MAE :", np.mean(mae_scores))
print("RMSE:", np.mean(rmse_scores))
print("R2  :", np.mean(r2_scores))

In [ ]:
#how yesterday's info relates to today's values.
df['humidity_lag_1']=df['humidity'].shift(1)
df['wind_speed_lag_1']=df['wind_speed'].shift(1)
df['meanpressure_lag_1']=df['meanpressure'].shift(1)
print(df.columns)

print(df.isnull().sum())
df=df.dropna()

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score


mae_scores=[]
rmse_scores=[]
r2_scores=[]

X=df.drop(columns=['meantemp','humidity','wind_speed','meanpressure'])
y=target

for i,(train_index,test_index) in enumerate(tscv.split(X)):

  x_train=X.iloc[train_index]
  y_train=y.iloc[train_index]

  x_test=X.iloc[test_index]
  y_test=y.iloc[test_index]

  xgb_model=xgb_reg.fit(x_train,y_train)
  y_pred=xgb_model.predict(x_test)

  mae=mean_absolute_error(y_test,y_pred)
  rmse=root_mean_squared_error(y_test,y_pred)
  r2=r2_score(y_test,y_pred)

  mae_scores.append(mae)
  rmse_scores.append(rmse)
  r2_scores.append(r2)

  print(f"Fold {i+1}")
  print("MAE:", mae)
  print("RMSE:", rmse)
  print("R2:", r2)
  print("-"*30)

print("\nAverage Results")
print("MAE :", np.mean(mae_scores))
print("RMSE:", np.mean(rmse_scores))
print("R2  :", np.mean(r2_scores))